In [0]:
%sql
CREATE OR REPLACE TEMP VIEW employees AS
SELECT * 
FROM VALUES 
    (1,  'Alice',   'Engineering', 95000),
    (2,  'Bob',     'Engineering', 88000),
    (3,  'Charlie', 'Engineering', 88000),
    (4,  'David',   'Engineering', 102000),
    (7,  'Grace',   'Sales', 72000),
    (8,  'Heidi',   'Sales', 91000),
    (9,  'Ivan',    'Sales', 58000),
    (12, 'Liam',    'HR', 55000),
    (13, 'Mia',     'HR', 61000),
    (16, 'Paul',    'HR', 65000),
    (17, 'Quinn',   'Marketing', 80000),
    (18, 'Ruth',    'Marketing', 80000),
    (19, 'Sam',     'Marketing', 95000),
    (20, 'Tina',    'Marketing', 73000),
    (20, 'Test',    'Marketing', NULL)
AS employees(emp_id, emp_name, department, salary);

In [0]:
%sql
select * from employees

In [0]:
%sql
select emp_id, emp_name, department, salary
    from employees e1
    where (select count(*) from employees e2 where e2.department = e1.department
    and e1.salary < e2.salary
    )<3

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW employees AS
SELECT *
FROM VALUES
    (1, 'Alice',   'Engineering', 120000),
    (2, 'Bob',     'Engineering', 110000),
    (3, 'Charlie', 'Engineering', 110000),
    (4, 'David',   'Engineering', 90000),
    (5, 'Eva',     'Engineering', NULL),

    (6, 'Frank',   'Sales', 95000),
    (7, 'Grace',   'Sales', 85000),
    (8, 'Heidi',   'Sales', 70000),
    (9, 'Ivan',    'Sales', 60000),

    (10, 'Jack',   'HR', 50000),
    (11, 'Kate',   'HR', 55000),
    (12, 'Liam',   'HR', 80000),

    (13, 'Mia',    'Marketing', 105000),
    (14, 'Nick',   'Marketing', 85000),
    (15, 'Olivia', 'Marketing', 80000)

AS employees(emp_id, emp_name, department, salary);

In [0]:
%sql
select * from employees

In [0]:
%sql
 with dept_sal_stat as(
 select department, max(salary) as max_sal, avg(salary) as avg_sal from employees group by all
)
select * from dept_sal_stat
    

In [0]:
%sql
 with dept_sal_stat as(
 select department, max(salary) as max_sal, avg(salary) as avg_sal from employees group by all
)
select e.* from employees e join dept_sal_stat d where e.department = d.department and e.salary < d.max_sal and e.salary > d.avg_sal
    

In [0]:
%sql
 with dept_sal_stat as(
 select department, max(salary) as max_sal, avg(salary) as avg_sal from employees group by all
)
select * from employees e where exists (select 1 from dept_sal_stat d where e.department = d.department and e.salary < d.max_sal and e.salary > d.avg_sal)
    

In [0]:
%sql
SELECT *
FROM employees e1
WHERE salary >
(
    SELECT AVG(salary)
    FROM employees e2
    WHERE e2.department = e1.department
)
AND salary <
(
    SELECT MAX(salary)
    FROM employees e3
    WHERE e3.department = e1.department
);

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW customers AS
SELECT *
FROM VALUES
    (101, 'Amazon'),
    (102, 'Google'),
    (103, 'Netflix'),
    (104, 'Uber'),
    (105, 'Airbnb')
AS customers(customer_id, customer_name);

CREATE OR REPLACE TEMP VIEW orders AS
SELECT *
FROM VALUES
    (1, 101, '2025-01-01'),
    (2, 101, '2025-02-01'),
    (3, 102, '2025-03-01'),
    (4, 102, '2025-04-01'),
    (5, 103, '2025-01-15'),
    (6, 101, '2025-03-01'),
    (7, 101, '2025-04-01')
AS orders(order_id, customer_id, order_date);

In [0]:
%sql
select * from customers

In [0]:
%sql
select * from orders

In [0]:
%sql
with cust_month_pair as (

    select distinct customer_id, date_format(order_date, 'yyyy-MM') as year_month from orders
),
total_months as (select count(distinct year_month) from cust_month_pair)
select * from  customers c where 
(select count(*) from cust_month_pair cm where c.customer_id = cm.customer_id) = (select * from total_months)


In [0]:
%sql
with month_ct as (

    select count(distinct date_format(order_date, 'yyyy-MM')) as month_ct from orders
),
cust_momnth_pair as (
select distinct customer_name,  date_format(order_date, 'yyyy-MM') as year_month from customers c join orders o on o.customer_id = c.customer_id 
)
select customer_name from cust_momnth_pair
group by all
having count(*) = (select month_ct from month_ct)


### Find products whose price is greater than at least 75% of products in their category.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW products AS
SELECT *
FROM VALUES
    (1, 'iPhone',      'Mobile', 90000),
    (2, 'Samsung S25', 'Mobile', 85000),
    (3, 'Vivo 10',    'Mobile', 40000),
    (4, 'Oppo 10',    'Mobile', 35000),
    (5, 'Pixel 10',    'Mobile', 70000),

    (4, 'Macbook',     'Laptop', 160000),
    (5, 'Dell XPS',    'Laptop', 120000),
    (6, 'ThinkPad',    'Laptop', 100000),

    (7, 'Sony TV',     'TV', 120000),
    (8, 'LG TV',       'TV', 100000),
    (9, 'TCL TV',      'TV', 70000)

AS products(product_id, product_name, category, price);

In [0]:
%sql
select * from products

In [0]:
%sql
with catagory_ct as (

    select category, count(*) as ct from products group by all
)
select product_name, prd.category, prd_ct/ct.ct as ratio from
(select p1.product_name,p1.category, count(p1.product_id) as prd_ct from products p1
join products p2 on p1.category = p2.category
and p1.price >= p2.price
group by all ) prd
join catagory_ct ct on prd.category = ct.category
where prd_ct/ct.ct>0.75

In [0]:
%sql
with catagory_ct as (

    select category, count(*) as ct from products group by all
)
select p1.product_name,p1.category from products p1
join catagory_ct ct on p1.category = ct.category
where ( select count(*)  from
 products p2 where p1.category = p2.category
and p1.price >= p2.price
)/ct.ct>0.75



### Return employees whose salary is greater than the average salary of employees earning less than them within the same department.

- It does not make sense: bcz Return employees whose salary is greater than the average salary of employees earning less than them within the same department.--- Isn't it very obivious.. Lets think- average salary of employees earning less than them within the same department. : If the salary is less then avg will alos be less.. then the all the rows will be true

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW employees AS
SELECT * 
FROM VALUES 
    (1,  'Alice',   'Engineering', 95000),
    (2,  'Bob',     'Engineering', 88000),
    (3,  'Charlie', 'Engineering', 88000),
    (4,  'David',   'Engineering', 102000),
    (7,  'Grace',   'Sales', 72000),
    (8,  'Heidi',   'Sales', 91000),
    (9,  'Ivan',    'Sales', 58000),
    (12, 'Liam',    'HR', 55000),
    (13, 'Mia',     'HR', 61000),
    (16, 'Paul',    'HR', 65000),
    (17, 'Quinn',   'Marketing', 80000),
    (18, 'Ruth',    'Marketing', 80000),
    (19, 'Sam',     'Marketing', 95000),
    (20, 'Tina',    'Marketing', 73000),
    (20, 'Test',    'Marketing', NULL)
AS employees(emp_id, emp_name, department, salary);

In [0]:
%sql
select * from employees 

In [0]:
%sql
with emply_avg_sal as (

    select e1.emp_id, e1.emp_name, e1.department, e1.salary, avg(e2.salary) as avg_sal  from employees e1
    left join employees e2 on e1.department = e2.department
    and e1.salary > e2.salary
    group by all
)
select * from emply_avg_sal

In [0]:
%sql
with ranked_sal as (

    select emp_id, emp_name, department, salary, row_number() over (partition by department order by salary desc NULLS last) as rn
    from employees
)

select emp_id, emp_name, department, salary, rn
from ranked_sal

where rn <= 3